In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('DateFruit_Dataset.csv')

In [4]:
df.head() # 34 features (input) and 1 category (output)

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB         898 no

In [6]:
df.shape

(898, 35)

In [7]:
X = df.drop('Class', axis=1)
y = df['Class']

In [8]:
y.head

<bound method NDFrame.head of 0      BERHI
1      BERHI
2      BERHI
3      BERHI
4      BERHI
       ...  
893    SOGAY
894    SOGAY
895    SOGAY
896    SOGAY
897    SOGAY
Name: Class, Length: 898, dtype: str>

In [9]:
df['Class'].value_counts()

Class
DOKOL     204
SAFAVI    199
ROTANA    166
DEGLET     98
SOGAY      94
IRAQI      72
BERHI      65
Name: count, dtype: int64

In [10]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder() # 7 types of values in class column, so label encoding
y = le.fit_transform(y)

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### Deep Learning

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [14]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # target values always in long dtype

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [15]:
# Create tensor dataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [16]:
# create data loader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [17]:
# Architecture 
# 34 Features (X)
# 2 Hidden Layer 
# 64 Neurons in each hidden layer
# 7 neurons in the Output layer

In [18]:
# Build Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64), # linear is the activation func
            nn.ReLU(), ## hidden layer 1

            nn.Linear(64,64), # hidden layer 2
            nn.ReLU(),
            
            nn.Linear(64,7) # output layer
        )

# forward propagation
    def forward(self, x):
        return self.model(x)

In [19]:
model = ANN()

# define loss and optimizer
criteria = nn.CrossEntropyLoss() # loss func
optimizer = optim.Adam(model.parameters()) # optimizer

In [20]:
# Training the neural network

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0 # training runningloss

    for xb,yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update
        running_loss += loss.item()

    # this is avg loss per batch in 1 Epoch
    train_loss = running_loss/len(train_loader)
    print(f'Epoch = {epoch+1}/{epochs}, loss = {train_loss}')

Epoch = 1/100, loss = 1.7079962491989136
Epoch = 2/100, loss = 1.0859726745149363
Epoch = 3/100, loss = 0.6780133376950803
Epoch = 4/100, loss = 0.4979282837847005
Epoch = 5/100, loss = 0.41769304353257886
Epoch = 6/100, loss = 0.35514666949925217
Epoch = 7/100, loss = 0.3227441194264785
Epoch = 8/100, loss = 0.29714022965534875
Epoch = 9/100, loss = 0.27222224130578665
Epoch = 10/100, loss = 0.2543072344168373
Epoch = 11/100, loss = 0.23037905246019363
Epoch = 12/100, loss = 0.21888129445521728
Epoch = 13/100, loss = 0.2162642012471738
Epoch = 14/100, loss = 0.19408617103877274
Epoch = 15/100, loss = 0.18885496777036917
Epoch = 16/100, loss = 0.18433331341847128
Epoch = 17/100, loss = 0.17574099127365195
Epoch = 18/100, loss = 0.16680864957363709
Epoch = 19/100, loss = 0.16700019071931424
Epoch = 20/100, loss = 0.16248842944269595
Epoch = 21/100, loss = 0.15507045566387798
Epoch = 22/100, loss = 0.14453778723659721
Epoch = 23/100, loss = 0.1471539450728375
Epoch = 24/100, loss = 0.141

In [21]:
# Evaluate the model

model.eval() # evaluation mode

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2, 0.5, 1.3 etc - total 7]
        _, precited = torch.max(outputs, 1)

        correct += (precited == yb).sum().item()
        total += yb.size(0) # return actaual samples in each batch

print('Total Values: ', total)
print('Correct Values: ', correct)
print('Accuracy Score: ', correct/total*100)

Total Values:  180
Correct Values:  170
Accuracy Score:  94.44444444444444


#### Further Advancement
- Apply PCA for 34 features and do Dimensionality Reduction